# About

This notebook is for testing if the dataset cleaning was proper or not, as I encountered that a significant amount of data was removed from the dataset after cleaning.

## Imports

In [1]:
import pandas as pd
from src.preprocessing.validator import validate_schema, validate_rules
from src.preprocessing.result import ValidationResult
from src.core.config import load_validation_schema, load_validation_rules, load_cleaning_configuration
from src.preprocessing.cleaning import clean
import os
from pathlib import Path

## Loading Dataset

In [2]:
dataset_dir = '../dataset'
files = os.listdir(dataset_dir)
dataset_files = []

for file in files:
    if '.csv' in file:
        dataset_files.append(file)

df = pd.DataFrame()

for file in dataset_files:
    temp_df = pd.read_csv(Path(dataset_dir, file))

    df = pd.concat([df, temp_df])

In [3]:
df.columns = df.columns.str.strip()
df.drop_duplicates(inplace=True)
pd.set_option("display.float_format", "{:.2f}".format)
df = df.reset_index(drop=True)

In [4]:
df.drop(columns=["Fwd Header Length.1"], inplace=True)

# doing the above because the column is redundant.

## Validation & Cleaning

In [5]:
sp = Path('../config/preprocessing/validation_shema.yaml')
schema_cfg = load_validation_schema(sp)

schema_result = validate_schema(df = df, schema_cfg=schema_cfg)

rp = Path('../config/preprocessing/validation_rules.yaml')
rules_cfg = load_validation_rules(rp)
rule_result = validate_rules(df=df, rules_cfg=rules_cfg)

cleaning_cfg = load_cleaning_configuration('../config/preprocessing/cleaning.yaml')

validation_result = ValidationResult(schema_result, rule_result, cleaning_cfg, rules_cfg)

df_cleaned = clean(validation_result, df, cleaning_cfg)

schema_result_post_clean = validate_schema(df_cleaned, schema_cfg)
rule_result_post_clean = validate_rules(df_cleaned, rules_cfg)

validation_result = ValidationResult(schema_result_post_clean, rule_result_post_clean, cleaning_cfg, rules_cfg)

df_cleaned_final = df_cleaned.drop(index=validation_result.non_repairable)

schema_result_final = validate_schema(df_cleaned_final, schema_cfg)
rule_result_final = validate_rules(df_cleaned_final, rules_cfg)

In [10]:
print(schema_result_final)
print('\n\n\n')
print(rule_result_final)

Schema Validation Result
-----------------
NaN:  0
Inf:  0
Negative:  0
Out of range values:  0
-----------------
NOTE: These are number of cell values not rows.




R001: 0
R002: 0
R003: 0
R004: 0
R005: 0
R006: 0
R007: 0
R008: 0
R009: 0
R010: 0
R011: 0
R012: 0
R013: 0
R014: 0
R015: 0
R016: 0
R017: 0
R018: 0
R019: 0



## Comparison

In [11]:
df_comparison = pd.DataFrame({
    'Original': [],
    'First Clean': [],
    'First Remove': [],
    'First remove percent': [],
    'Second Remove': [],
    'Second Removed': [],
    'Second Removed Percent': []
})

df_comparison['Original'] = df['Label'].value_counts()
df_comparison['First Clean'] = df_cleaned['Label'].value_counts()
df_comparison['First Remove'] = df['Label'].value_counts() - df_cleaned['Label'].value_counts()
df_comparison['First remove percent'] = 100 -(df_cleaned['Label'].value_counts() / df['Label'].value_counts()) * 100
df_comparison['Second Clean'] = df_cleaned_final['Label'].value_counts()
df_comparison['Second Remove'] = df_cleaned['Label'].value_counts() - df_cleaned_final['Label'].value_counts()
df_comparison['Second remove percent'] =  100 - (df_cleaned_final['Label'].value_counts() / df_cleaned['Label'].value_counts()) * 100

In [12]:
comp_path = Path('../out/data/')
comp_path.mkdir(parents=True, exist_ok=True)
df_comparison.to_csv(Path(comp_path, 'df_comparison.csv'))

In [13]:
df_comparison

,Original,First Clean,First Remove,First remove percent,Second Remove,Second Removed,Second Removed Percent,Second Clean,Second remove percent
Label,,,,,,,,,
BENIGN,2096484,531131.00,1565353.00,74.67,0.00,NaN,NaN,531131.00,0.00
DoS Hulk,172849,8662.00,164187.00,94.99,0.00,NaN,NaN,8662.00,0.00
DDoS,128016,93.00,127923.00,99.93,0.00,NaN,NaN,93.00,0.00
PortScan,90819,90408.00,411.00,0.45,0.00,NaN,NaN,90408.00,0.00
DoS GoldenEye,10286,81.00,10205.00,99.21,0.00,NaN,NaN,81.00,0.00
FTP-Patator,5933,5864.00,69.00,1.16,0.00,NaN,NaN,5864.00,0.00
DoS slowloris,5385,3177.00,2208.00,41.00,0.00,NaN,NaN,3177.00,0.00
DoS Slowhttptest,5228,1123.00,4105.00,78.52,0.00,NaN,NaN,1123.00,0.00
SSH-Patator,3219,3187.00,32.00,0.99,0.00,NaN,NaN,3187.00,0.00


From the above I can say for certain that the cleaning  was either wrong or too aggressive and removed a lot of data.